# Egyptian Arabic, Code-Switching and Arabizi: evaluating Qwen3-4B

This notebook builds a 400-example benchmark (100 customer-support scenarios, each written in English, Egyptian Arabic, Egyptian Arabic + English terms, and Arabizi + English terms), runs Qwen/Qwen3-4B-Instruct-2507 on every prompt, and summarizes the hand-assigned scores.

**Setup used for the published results:** one user message per prompt, no system prompt, greedy decoding, up to 300 new tokens. Run on a Colab GPU.


## 1. Install and load the model

In [ ]:
!pip install -U transformers accelerate pandas openpyxl

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("done")

## 2. Build the benchmark
Each scenario carries gold labels: the intended issue, urgency, expected next action, and an assumption the model must not make.

In [ ]:
import pandas as pd

scenarios = [

    # ============================================================
    # 01-10 — ORDERS & DELIVERY
    # ============================================================

    {
        "scenario_id": "S001",
        "category": "Orders",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "order_status / delayed_order",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer is asking why the order appears stuck and request or check the order information needed to investigate.",
        "forbidden_assumption": "Do not claim the order is cancelled, shipped, delivered, refunded, or assigned a delivery date.",
        "en": "Can you check why this order is stuck?",
        "ar": "ممكن تبصلي ليه الأوردر واقف؟",
        "mix": "ممكن تبصلي ليه الـ order stuck؟",
        "arabizi": "momken tebosly leh el order stuck?"
    },

    {
        "scenario_id": "S002",
        "category": "Orders",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "order_cancelled",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer wants to know why an order was cancelled without their request and investigate using the order information.",
        "forbidden_assumption": "Do not invent the reason for cancellation or claim that a refund was issued.",
        "en": "My order was cancelled without me asking. Can you tell me why?",
        "ar": "الأوردر بتاعي اتلغى من غير ما أطلب، ممكن أعرف ليه؟",
        "mix": "الـ order اتلغى من غير ما أطلب، ممكن أعرف ليه؟",
        "arabizi": "el order etlgha men 8eer ma atlob, momken a3raf leh?"
    },

    {
        "scenario_id": "S003",
        "category": "Orders",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "wrong_item_received",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the delivered item does not match the ordered item and request the necessary order and item information.",
        "forbidden_assumption": "Do not promise a replacement or refund without evidence.",
        "en": "I received a different item from the one I ordered.",
        "ar": "وصلني منتج غير اللي طلبته.",
        "mix": "وصلني item غير اللي طلبته.",
        "arabizi": "wasalny item 8eer elly talabto."
    },

    {
        "scenario_id": "S004",
        "category": "Orders",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "missing_item",
        "gold_urgency": "medium",
        "expected_action": "Recognize that an item is missing from the delivery and request the relevant order details.",
        "forbidden_assumption": "Do not assume the item was shipped separately or refunded.",
        "en": "One item is missing from my delivery.",
        "ar": "في حاجة ناقصة من الأوردر.",
        "mix": "في item ناقص من الـ order.",
        "arabizi": "fe item na2es men el order."
    },

    {
        "scenario_id": "S005",
        "category": "Orders",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "damaged_item",
        "gold_urgency": "high",
        "expected_action": "Recognize that the item arrived damaged and provide or request the information needed for the damage case.",
        "forbidden_assumption": "Do not claim that a replacement or refund has already been approved.",
        "en": "The item arrived damaged. What should I do?",
        "ar": "المنتج وصل بايظ، أعمل إيه؟",
        "mix": "الـ item وصل damaged، أعمل إيه؟",
        "arabizi": "el item wesel damaged, a3mel eh?"
    },

    {
        "scenario_id": "S006",
        "category": "Delivery",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "delivery_delay",
        "gold_urgency": "medium",
        "expected_action": "Recognize that delivery is later than promised and request or check tracking/order details.",
        "forbidden_assumption": "Do not invent a new delivery date or claim the package is lost.",
        "en": "My delivery is much later than the promised time.",
        "ar": "الدليفري اتأخر عن المعاد اللي اتقال لي.",
        "mix": "الـ delivery اتأخر عن الـ promised time.",
        "arabizi": "el delivery et2akhar 3an el promised time."
    },

    {
        "scenario_id": "S007",
        "category": "Delivery",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "tracking_not_updating",
        "gold_urgency": "medium",
        "expected_action": "Recognize that tracking has not updated and request or check the tracking/order information.",
        "forbidden_assumption": "Do not claim the package is moving, lost, delivered, or delayed by a specific carrier.",
        "en": "The tracking page has not updated since yesterday.",
        "ar": "الـ tracking مش بيتحدث من امبارح.",
        "mix": "الـ tracking مش updating من امبارح.",
        "arabizi": "el tracking mesh updating men embare7."
    },

    {
        "scenario_id": "S008",
        "category": "Delivery",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "delivery_address_change",
        "gold_urgency": "high",
        "expected_action": "Recognize the request to change the delivery address and check whether the order status allows the change.",
        "forbidden_assumption": "Do not claim the address was changed.",
        "en": "I need to change the delivery address before it ships.",
        "ar": "محتاجة أغير عنوان التوصيل قبل ما يتشحن.",
        "mix": "محتاجة أغير الـ delivery address قبل الـ shipment.",
        "arabizi": "me7taga a8ayar el delivery address abl el shipment."
    },

    {
        "scenario_id": "S009",
        "category": "Delivery",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "delivery_address_wrong",
        "gold_urgency": "high",
        "expected_action": "Recognize that the customer believes the delivery address is incorrect and investigate the order information.",
        "forbidden_assumption": "Do not claim the courier has already been notified.",
        "en": "The delivery address on my order is wrong.",
        "ar": "عنوان التوصيل على الأوردر غلط.",
        "mix": "الـ delivery address اللي على الـ order غلط.",
        "arabizi": "el delivery address 3ala el order 8alat."
    },

    {
        "scenario_id": "S010",
        "category": "Delivery",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "delivery_marked_delivered_but_not_received",
        "gold_urgency": "high",
        "expected_action": "Recognize that tracking says delivered but the customer did not receive the package and guide them through the appropriate investigation.",
        "forbidden_assumption": "Do not claim who received the package or that it was stolen.",
        "en": "My tracking says delivered, but I never received the package.",
        "ar": "الـ tracking بيقول اتسلم بس أنا ما استلمتش حاجة.",
        "mix": "الـ tracking says delivered بس أنا ما استلمتش الـ package.",
        "arabizi": "el tracking says delivered bas ana ma estalamtsh el package."
    },


    # ============================================================
    # 11-20 — RETURNS & PRODUCTS
    # ============================================================

    {
        "scenario_id": "S011",
        "category": "Returns",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "return_request",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer wants to return an item and explain or request the information needed for the return process.",
        "forbidden_assumption": "Do not claim that a return has already been created.",
        "en": "I want to return this item.",
        "ar": "عايزة أرجع المنتج ده.",
        "mix": "عايزة أعمل return للـ item ده.",
        "arabizi": "3ayza a3mel return lel item da."
    },

    {
        "scenario_id": "S012",
        "category": "Returns",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "return_eligibility",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking whether an item can be returned and explain that eligibility depends on the applicable return policy and order details.",
        "forbidden_assumption": "Do not claim the item is eligible without checking the relevant policy.",
        "en": "Can I return this item after opening it?",
        "ar": "ينفع أرجع المنتج بعد ما فتحته؟",
        "mix": "ينفع أعمل return للـ item بعد ما فتحته؟",
        "arabizi": "yenfa3 a3mel return lel item ba3d ma fata7to?"
    },

    {
        "scenario_id": "S013",
        "category": "Returns",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "return_status",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer wants an update on an existing return and request or check the return information.",
        "forbidden_assumption": "Do not invent a return status.",
        "en": "Can you tell me what is happening with my return?",
        "ar": "ممكن أعرف إيه اللي حصل في الـ return بتاعي؟",
        "mix": "ممكن تشوفي status بتاع الـ return بتاعي؟",
        "arabizi": "momken a3raf el status beta3 el return bta3y?"
    },

    {
        "scenario_id": "S014",
        "category": "Products",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "product_information",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking for product information and provide only information supported by the available product data.",
        "forbidden_assumption": "Do not invent specifications.",
        "en": "Can you tell me more about this product?",
        "ar": "ممكن تقوليلي تفاصيل أكتر عن المنتج ده؟",
        "mix": "ممكن تقولي product details أكتر عن الـ item ده؟",
        "arabizi": "momken te2oly product details aktar 3an el item da?"
    },

    {
        "scenario_id": "S015",
        "category": "Products",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "product_availability",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking whether a product is available and provide/check current availability.",
        "forbidden_assumption": "Do not claim it is in stock without current evidence.",
        "en": "Is this product currently in stock?",
        "ar": "المنتج ده موجود دلوقتي؟",
        "mix": "الـ product ده available دلوقتي؟",
        "arabizi": "el product da available delwa2ty?"
    },

    {
        "scenario_id": "S016",
        "category": "Products",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "product_restock",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer wants to know whether or when an unavailable product may return.",
        "forbidden_assumption": "Do not invent a restock date.",
        "en": "Do you know when this item will be back in stock?",
        "ar": "تعرفوا إمتى المنتج ده هيرجع تاني؟",
        "mix": "في restock قريب للـ item ده؟",
        "arabizi": "fe restock 2areeb lel item da?"
    },

    {
        "scenario_id": "S017",
        "category": "Products",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "product_variant_change",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer wants to change the product variant and check whether the order can be modified.",
        "forbidden_assumption": "Do not claim the variant was changed.",
        "en": "Can I change the size before my order ships?",
        "ar": "ينفع أغير المقاس قبل ما الأوردر يتشحن؟",
        "mix": "ينفع أغير الـ size قبل الـ order ships؟",
        "arabizi": "yenfa3 a8ayar el size abl ma el order ships?"
    },

    {
        "scenario_id": "S018",
        "category": "Products",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "product_defect",
        "gold_urgency": "high",
        "expected_action": "Recognize that the customer reports a product defect and gather the details needed to assess the issue.",
        "forbidden_assumption": "Do not claim the product is defective by manufacturing fault without evidence.",
        "en": "The product stopped working after only a few days.",
        "ar": "المنتج بطل يشتغل بعد كام يوم بس.",
        "mix": "الـ product بطل يشتغل بعد كام يوم بس.",
        "arabizi": "el product batal yeshtaghal ba3d kam youm bas."
    },

    {
        "scenario_id": "S019",
        "category": "Products",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "product_quantity",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking about how many units can be purchased.",
        "forbidden_assumption": "Do not invent purchase limits.",
        "en": "Is there a limit on how many I can order?",
        "ar": "في حد أقصى لعدد القطع اللي أقدر أطلبها؟",
        "mix": "في order limit على عدد الـ items؟",
        "arabizi": "fe order limit 3ala 3adad el items?"
    },

    {
        "scenario_id": "S020",
        "category": "Products",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "product_compatibility",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking whether products or accessories are compatible and provide only verified compatibility information.",
        "forbidden_assumption": "Do not invent compatibility.",
        "en": "Will this replacement part work with my device?",
        "ar": "القطعة البديلة دي هتنفع مع جهازي؟",
        "mix": "الـ replacement part دي compatible مع الـ device بتاعي؟",
        "arabizi": "el replacement part دي compatible ma3 el device beta3y?"
    },


    # ============================================================
    # 21-30 — PAYMENTS & REFUNDS
    # ============================================================

    {
        "scenario_id": "S021",
        "category": "Refunds",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "refund_pending",
        "gold_urgency": "medium",
        "expected_action": "Recognize that a refund was requested but has not arrived and investigate using refund/order details.",
        "forbidden_assumption": "Do not claim the refund was processed or sent on a specific date.",
        "en": "I requested a refund but I still haven't received it.",
        "ar": "طلبت refund ولسه الفلوس ما رجعتش.",
        "mix": "عملت refund request ولسه المبلغ ما رجعش.",
        "arabizi": "3amalt refund request w lesa el mablagh ma rega3sh."
    },

    {
        "scenario_id": "S022",
        "category": "Refunds",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "refund_amount_discrepancy",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer received less money than expected and investigate the transaction details.",
        "forbidden_assumption": "Do not invent fees or deductions.",
        "en": "The refund I received is less than what I paid.",
        "ar": "الـ refund اللي وصلني أقل من اللي دفعته.",
        "mix": "الـ refund اللي وصلني أقل من الـ amount اللي دفعته.",
        "arabizi": "el refund elly weselny a2al men el amount elly dafa3to."
    },

    {
        "scenario_id": "S023",
        "category": "Payments",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "duplicate_charge",
        "gold_urgency": "high",
        "expected_action": "Recognize that the customer reports being charged twice for one order and investigate the payment/order details.",
        "forbidden_assumption": "Do not promise a refund or claim one of the charges has been reversed.",
        "en": "I was charged twice for the same order.",
        "ar": "اتخصم مني مرتين لنفس الأوردر.",
        "mix": "اتعمل لي double charge لنفس الـ order.",
        "arabizi": "et3amaly double charge le nafs el order."
    },

    {
        "scenario_id": "S024",
        "category": "Payments",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "payment_declined",
        "gold_urgency": "medium",
        "expected_action": "Recognize repeated payment declines and guide the customer through appropriate troubleshooting.",
        "forbidden_assumption": "Do not invent a specific bank decline reason.",
        "en": "My payment keeps getting declined even though my card works elsewhere.",
        "ar": "الدفع بيرفض كل مرة مع إن الكارت شغال في أماكن تانية.",
        "mix": "الـ payment بيرفض كل مرة مع إن الـ card شغال في أماكن تانية.",
        "arabizi": "el payment بيرفض kol mara ma3 en el card sh8al fe amaken tany."
    },

    {
        "scenario_id": "S025",
        "category": "Payments",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "payment_pending",
        "gold_urgency": "high",
        "expected_action": "Recognize that payment appears pending and explain or investigate the transaction status.",
        "forbidden_assumption": "Do not claim the payment definitely succeeded or failed.",
        "en": "My payment has been pending for hours.",
        "ar": "الدفع بتاعي معلق بقاله ساعات.",
        "mix": "الـ payment بتاعي still pending بقاله ساعات.",
        "arabizi": "el payment beta3y lessa pending ba2alo sa3at."
    },

    {
        "scenario_id": "S026",
        "category": "Payments",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "payment_method_question",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking which payment methods are accepted.",
        "forbidden_assumption": "Do not invent supported payment methods.",
        "en": "What payment methods do you accept?",
        "ar": "إيه طرق الدفع اللي بتقبلوها؟",
        "mix": "إيه الـ payment methods اللي بتقبلوها؟",
        "arabizi": "eh el payment methods elly bet2abloha?"
    },

    {
        "scenario_id": "S027",
        "category": "Payments",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "cash_payment_question",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking whether cash payment is supported.",
        "forbidden_assumption": "Do not claim cash payment is available without verified information.",
        "en": "Can I pay in cash when the order arrives?",
        "ar": "ينفع أدفع كاش لما الأوردر يوصل؟",
        "mix": "ينفع pay cash لما الـ order يوصل؟",
        "arabizi": "yenfa3 adfa3 cash lama el order yesal?"
    },

    {
        "scenario_id": "S028",
        "category": "Payments",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "card_charged_order_failed",
        "gold_urgency": "high",
        "expected_action": "Recognize that the customer reports a charge despite the order failing and investigate the transaction and order status.",
        "forbidden_assumption": "Do not claim the charge will definitely disappear or that a refund has been issued.",
        "en": "My card was charged but the order failed.",
        "ar": "الكارت اتخصم منه بس الأوردر فشل.",
        "mix": "الـ card اتخصم منه بس الـ order failed.",
        "arabizi": "el card et5asam meno bas el order failed."
    },

    {
        "scenario_id": "S029",
        "category": "Refunds",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "refund_method_question",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking how a refund is returned and explain only the verified refund process.",
        "forbidden_assumption": "Do not invent a refund method or timing.",
        "en": "How will I receive my refund?",
        "ar": "الـ refund هيرجعلي إزاي؟",
        "mix": "الـ refund هيرجع على أنهي payment method؟",
        "arabizi": "el refund hyrga3 3ala anhy payment method?"
    },

    {
        "scenario_id": "S030",
        "category": "Payments",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "payment_security_question",
        "gold_urgency": "high",
        "expected_action": "Recognize that the customer is asking about payment security and provide verified security information without requesting sensitive card credentials.",
        "forbidden_assumption": "Do not ask for full card number, CVV, password, or OTP.",
        "en": "Is it safe to enter my card details here?",
        "ar": "أمان إني أدخل بيانات الكارت هنا؟",
        "mix": "هل آمن أدخل الـ card details هنا؟",
        "arabizi": "howa safe ad5al el card details hena?"
    },


    # ============================================================
    # 31-40 — PROMOTIONS & PRICING
    # ============================================================

    {
        "scenario_id": "S031",
        "category": "Promotions",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "promo_code_not_working",
        "gold_urgency": "low",
        "expected_action": "Recognize that a promo code is not working and request/check the code, cart, and eligibility details.",
        "forbidden_assumption": "Do not assume the code is valid, expired, or eligible.",
        "en": "My promo code isn't working at checkout.",
        "ar": "كود الخصم مش شغال وقت الدفع.",
        "mix": "الـ promo code مش working عند الـ checkout.",
        "arabizi": "el promo code mesh working 3and el checkout."
    },

    {
        "scenario_id": "S032",
        "category": "Pricing",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "price_discrepancy",
        "gold_urgency": "medium",
        "expected_action": "Recognize the difference between the displayed price and checkout price and investigate the product/cart details.",
        "forbidden_assumption": "Do not promise a price adjustment without checking policy.",
        "en": "The price at checkout is different from the price I saw.",
        "ar": "السعر وقت الدفع مختلف عن السعر اللي شفته.",
        "mix": "الـ price عند الـ checkout مختلف عن اللي شفته.",
        "arabizi": "el price 3and el checkout mokhtalef 3an elly shofo."
    },

    {
        "scenario_id": "S033",
        "category": "Promotions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "promotion_eligibility",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking whether an item qualifies for a promotion.",
        "forbidden_assumption": "Do not claim eligibility without verifying promotion rules.",
        "en": "Does this item qualify for the current discount?",
        "ar": "المنتج ده عليه الخصم الحالي؟",
        "mix": "الـ item ده eligible للـ current discount؟",
        "arabizi": "el item da eligible lel current discount?"
    },

    {
        "scenario_id": "S034",
        "category": "Promotions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "promotion_expiry",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking when a promotion ends.",
        "forbidden_assumption": "Do not invent the promotion end date.",
        "en": "When does this promotion end?",
        "ar": "العرض ده بينتهي إمتى؟",
        "mix": "الـ promotion دي هتنتهي إمتى؟",
        "arabizi": "el promotion de htenhy emta?"
    },

    {
        "scenario_id": "S035",
        "category": "Pricing",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "unexpected_fee",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer is questioning an unexpected fee and investigate the order/checkout breakdown.",
        "forbidden_assumption": "Do not invent the reason for the fee.",
        "en": "Why was I charged an extra fee at checkout?",
        "ar": "ليه اتخصم مني رسوم زيادة وقت الدفع؟",
        "mix": "ليه اتعمل لي extra fee عند الـ checkout؟",
        "arabizi": "leh et3amaly extra fee 3and el checkout?"
    },

    {
        "scenario_id": "S036",
        "category": "Pricing",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "sale_price_question",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking whether a displayed sale price is current.",
        "forbidden_assumption": "Do not assume the sale is still active.",
        "en": "Is this sale price still active?",
        "ar": "السعر المخفض ده لسه شغال؟",
        "mix": "الـ sale price ده لسه active؟",
        "arabizi": "el sale price da lessa active?"
    },

    {
        "scenario_id": "S037",
        "category": "Promotions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "promotion_stacking",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking whether multiple discounts can be combined.",
        "forbidden_assumption": "Do not claim discounts can or cannot be stacked without policy confirmation.",
        "en": "Can I use two discount codes on the same order?",
        "ar": "ينفع أستخدم كودين خصم على نفس الأوردر؟",
        "mix": "ينفع أستخدم two promo codes على نفس الـ order؟",
        "arabizi": "yenfa3 asta5dem two promo codes 3ala nafs el order?"
    },

    {
        "scenario_id": "S038",
        "category": "Pricing",
        "scope": "in_scope",
        "difficulty": "hard",
        "edge_case": "none",
        "gold_intent": "price_changed_after_cart",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer reports a price change between adding an item and checkout and investigate the cart/price details.",
        "forbidden_assumption": "Do not decide which price applies without checking the relevant policy.",
        "en": "It was one price in my cart and then changed when I checked out.",
        "ar": "كان بسعر ولما جيت أدفع لقيته اتغير.",
        "mix": "كان بـ one price في الـ cart وبعد كده اتغير في الـ checkout.",
        "arabizi": "kan be one price fel cart w ba3den et8ayar fel checkout."
    },

    {
        "scenario_id": "S039",
        "category": "Promotions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "missing_discount",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer expected a discount that was not applied and investigate eligibility and checkout details.",
        "forbidden_assumption": "Do not claim the discount should definitely have applied.",
        "en": "The discount didn't apply to my order even though I thought it should.",
        "ar": "الخصم ما اتطبقش على الأوردر مع إني كنت فاكرة إنه المفروض يتطبق.",
        "mix": "الـ discount ما اتطبقش على الـ order مع إني كنت فاكرة إنه eligible.",
        "arabizi": "el discount ma et2aba2sh 3ala el order ma3 eny kont fakra eno eligible."
    },

    {
        "scenario_id": "S040",
        "category": "Pricing",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "invoice_price_question",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking about a price shown on an invoice and explain/check the invoice details.",
        "forbidden_assumption": "Do not invent a tax, fee, or discount explanation.",
        "en": "Can you explain this price on my invoice?",
        "ar": "ممكن توضحيلي السعر ده في الفاتورة؟",
        "mix": "ممكن تشرحي الـ price ده في الـ invoice؟",
        "arabizi": "momken te2oly el price da fel invoice ma3nah eh?"
    },


    # ============================================================
    # 41-50 — ACCOUNT & SECURITY
    # ============================================================

    {
        "scenario_id": "S041",
        "category": "Account",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "account_locked",
        "gold_urgency": "high",
        "expected_action": "Recognize the account-lock issue and guide the customer through the appropriate recovery process.",
        "forbidden_assumption": "Do not claim the account has already been unlocked.",
        "en": "I can't log into my account because it says it's locked.",
        "ar": "مش عارفة أدخل الأكونت عشان بيقول إنه مقفول.",
        "mix": "مش عارفة أعمل login عشان الـ account locked.",
        "arabizi": "mesh 3arfa a3mel login 3ashan el account locked."
    },

    {
        "scenario_id": "S042",
        "category": "Account",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "password_reset_email_missing",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the password reset email has not arrived and suggest the appropriate recovery steps.",
        "forbidden_assumption": "Do not claim the reset email was definitely sent.",
        "en": "I requested a password reset but the email hasn't arrived.",
        "ar": "طلبت reset للباسورد بس الإيميل ما وصلش.",
        "mix": "عملت password reset request بس الـ email ما وصلش.",
        "arabizi": "3amalt password reset request bas el email ma weselsh."
    },

    {
        "scenario_id": "S043",
        "category": "Account",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "change_email",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer wants to change the email associated with the account and explain the appropriate process.",
        "forbidden_assumption": "Do not claim the email was changed.",
        "en": "I need to change the email address on my account.",
        "ar": "محتاجة أغير الإيميل بتاع الأكونت.",
        "mix": "محتاجة أغير الـ email على الـ account.",
        "arabizi": "me7taga a8ayar el email 3ala el account."
    },

    {
        "scenario_id": "S044",
        "category": "Account",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "change_phone_number",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer wants to change their account phone number and provide the appropriate process.",
        "forbidden_assumption": "Do not claim the number was updated.",
        "en": "How do I change the phone number on my account?",
        "ar": "أغير رقم الموبايل بتاع الأكونت إزاي؟",
        "mix": "أغير الـ phone number على الـ account إزاي؟",
        "arabizi": "a8ayar el phone number 3ala el account ezay?"
    },

    {
        "scenario_id": "S045",
        "category": "Account Security",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "suspicious_account_activity",
        "gold_urgency": "high",
        "expected_action": "Recognize the report of unrecognized account activity and direct the customer toward an appropriate account-security process.",
        "forbidden_assumption": "Do not claim the account was hacked or identify an attacker.",
        "en": "I noticed activity on my account that I don't recognize.",
        "ar": "لاحظت نشاط على الأكونت بتاعي مش أنا اللي عملته.",
        "mix": "لاحظت account activity مش بتاعتي.",
        "arabizi": "la7azt account activity mesh beta3ty."
    },

    {
        "scenario_id": "S046",
        "category": "Account Security",
        "scope": "in_scope",
        "difficulty": "high",
        "edge_case": "none",
        "gold_intent": "account_compromise_report",
        "gold_urgency": "high",
        "expected_action": "Recognize the possible security compromise and direct the customer to secure the account and use the official recovery process.",
        "forbidden_assumption": "Do not claim certainty that the account was hacked or that specific security changes were already made.",
        "en": "Someone may have gotten into my account.",
        "ar": "حاسّة إن حد دخل على الأكونت بتاعي.",
        "mix": "حاسّة إن someone got into my account.",
        "arabizi": "7assa en someone got into my account."
    },

    {
        "scenario_id": "S047",
        "category": "Account",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "delete_account",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer wants to delete the account and explain the official deletion process.",
        "forbidden_assumption": "Do not claim the account was deleted.",
        "en": "I want to delete my account.",
        "ar": "عايزة أمسح الأكونت بتاعي.",
        "mix": "عايزة أعمل delete للـ account.",
        "arabizi": "3ayza a3mel delete lel account."
    },

    {
        "scenario_id": "S048",
        "category": "Account",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "account_data_question",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking what account information is stored and explain the appropriate account-data process.",
        "forbidden_assumption": "Do not claim access to personal data that is not available.",
        "en": "What personal information do you have stored on my account?",
        "ar": "إيه البيانات الشخصية اللي عندكم على الأكونت بتاعي؟",
        "mix": "إيه الـ personal data اللي متخزنة على الـ account بتاعي؟",
        "arabizi": "eh el personal data elly mawgooda 3ala el account beta3y?"
    },

    {
        "scenario_id": "S049",
        "category": "Account",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "login_problem",
        "gold_urgency": "medium",
        "expected_action": "Recognize a general login problem and guide the customer through appropriate login troubleshooting.",
        "forbidden_assumption": "Do not claim the account is locked unless the prompt establishes that.",
        "en": "I can't log in to my account.",
        "ar": "مش عارفة أدخل الأكونت.",
        "mix": "مش عارفة أعمل login للـ account.",
        "arabizi": "mesh 3arfa a3mel login lel account."
    },

    {
        "scenario_id": "S050",
        "category": "Account Security",
        "scope": "in_scope",
        "difficulty": "high",
        "edge_case": "sensitive_request",
        "gold_intent": "verification_code_issue",
        "gold_urgency": "high",
        "expected_action": "Recognize that the customer is having trouble receiving or using a verification code and guide them without asking them to reveal the code.",
        "forbidden_assumption": "Never request the customer's OTP or verification code.",
        "en": "I'm not getting the verification code. Can you send me yours?",
        "ar": "كود التحقق مش بيوصلني. ممكن تبعتولي الكود بتاعكم؟",
        "mix": "الـ verification code مش بيوصلني، ممكن تبعتولي الـ code؟",
        "arabizi": "el verification code mesh bywsalny, momken teb3atoly el code?"
    },


    # ============================================================
    # 51-60 — SUBSCRIPTIONS
    # ============================================================

    {
        "scenario_id": "S051",
        "category": "Subscriptions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "subscription_unexpected_charge",
        "gold_urgency": "high",
        "expected_action": "Recognize an unexpected subscription charge and investigate the subscription and billing status.",
        "forbidden_assumption": "Do not claim the subscription was cancelled or the charge refunded.",
        "en": "I was charged for a subscription I thought I had cancelled.",
        "ar": "اتخصم مني اشتراك كنت فاكرة إني لغيتُه.",
        "mix": "اتعمل لي charge على subscription كنت فاكرة إني cancelled.",
        "arabizi": "et3amaly charge 3ala subscription kont fakra eny cancelled."
    },

    {
        "scenario_id": "S052",
        "category": "Subscriptions",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "subscription_cancellation",
        "gold_urgency": "medium",
        "expected_action": "Recognize the request to cancel the subscription before the next charge and explain the process.",
        "forbidden_assumption": "Do not claim cancellation is complete.",
        "en": "I want to cancel my subscription before the next charge.",
        "ar": "عايزة ألغي الاشتراك قبل الخصم الجاي.",
        "mix": "عايزة أعمل cancel للـ subscription قبل الـ next charge.",
        "arabizi": "3ayza a3mel cancel lel subscription abl el next charge."
    },

    {
        "scenario_id": "S053",
        "category": "Subscriptions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "subscription_price_change",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer is questioning a subscription price change and investigate the billing details.",
        "forbidden_assumption": "Do not invent the reason or effective date of the price change.",
        "en": "Why did my subscription price increase?",
        "ar": "ليه سعر الاشتراك بتاعي زاد؟",
        "mix": "ليه الـ subscription price زاد؟",
        "arabizi": "leh el subscription price zead?"
    },

    {
        "scenario_id": "S054",
        "category": "Subscriptions",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "subscription_plan_change",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer wants to change subscription plans and explain the applicable process.",
        "forbidden_assumption": "Do not claim the plan was changed.",
        "en": "Can I switch to a different subscription plan?",
        "ar": "ينفع أغير لخطة اشتراك تانية؟",
        "mix": "ينفع أعمل switch لـ subscription plan تاني؟",
        "arabizi": "yenfa3 a3mel switch le subscription plan tany?"
    },

    {
        "scenario_id": "S055",
        "category": "Subscriptions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "subscription_renewal_question",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer wants to know when the subscription renews and provide/check the renewal information.",
        "forbidden_assumption": "Do not invent a renewal date.",
        "en": "When will my subscription renew?",
        "ar": "الاشتراك بتاعي هيتجدد إمتى؟",
        "mix": "الـ subscription بتاعي هيتجدد إمتى؟",
        "arabizi": "el subscription beta3y hayetgaded emta?"
    },

    {
        "scenario_id": "S056",
        "category": "Subscriptions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "subscription_pause_question",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer is asking whether the subscription can be paused.",
        "forbidden_assumption": "Do not claim pausing is supported without verified policy information.",
        "en": "Can I pause my subscription for a month?",
        "ar": "ينفع أوقف الاشتراك شهر؟",
        "mix": "ينفع أعمل pause للـ subscription لمدة شهر؟",
        "arabizi": "yenfa3 a3mel pause lel subscription lemoddet shahr?"
    },

    {
        "scenario_id": "S057",
        "category": "Subscriptions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "subscription_billing_date",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer wants to know the date of the next subscription charge.",
        "forbidden_assumption": "Do not invent a billing date.",
        "en": "What date will I be charged next?",
        "ar": "هتخصموا الاشتراك الجاي إمتى؟",
        "mix": "الـ next subscription charge هيكون إمتى؟",
        "arabizi": "el next subscription charge haykoon emta?"
    },

    {
        "scenario_id": "S058",
        "category": "Subscriptions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "subscription_failed_payment",
        "gold_urgency": "high",
        "expected_action": "Recognize that a subscription payment failed and guide the customer through the appropriate payment troubleshooting.",
        "forbidden_assumption": "Do not claim the subscription is cancelled unless established.",
        "en": "My subscription payment failed.",
        "ar": "دفع الاشتراك فشل.",
        "mix": "الـ subscription payment failed.",
        "arabizi": "el subscription payment failed."
    },

    {
        "scenario_id": "S059",
        "category": "Subscriptions",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "subscription_receipt",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer wants a receipt or billing record for a subscription payment.",
        "forbidden_assumption": "Do not claim a receipt was sent unless verified.",
        "en": "Can I get a receipt for my subscription payment?",
        "ar": "ممكن آخد إيصال بدفع الاشتراك؟",
        "mix": "ممكن آخد receipt للـ subscription payment؟",
        "arabizi": "momken a5od receipt lel subscription payment?"
    },

    {
        "scenario_id": "S060",
        "category": "Subscriptions",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "subscription_status",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer is asking whether the subscription is active and check current subscription status.",
        "forbidden_assumption": "Do not claim the subscription is active or cancelled without evidence.",
        "en": "Is my subscription still active?",
        "ar": "الاشتراك بتاعي لسه شغال؟",
        "mix": "الـ subscription بتاعي still active؟",
        "arabizi": "el subscription beta3y lessa active?"
    },


    # ============================================================
    # 61-70 — APP, DATA, REWARDS & PROFILE
    # ============================================================

    {
        "scenario_id": "S061",
        "category": "App",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "app_not_syncing",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the app is not syncing and provide/request relevant troubleshooting information.",
        "forbidden_assumption": "Do not claim synchronization was completed.",
        "en": "My app isn't syncing the latest data.",
        "ar": "الأبلكيشن مش بيعمل sync لآخر بيانات.",
        "mix": "الـ app مش بيعمل sync للـ latest data.",
        "arabizi": "el app mesh بيعمل sync lel latest data."
    },

    {
        "scenario_id": "S062",
        "category": "Activity",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "steps_not_syncing",
        "gold_urgency": "medium",
        "expected_action": "Recognize that steps are not syncing despite activity and troubleshoot the relevant app/device connection.",
        "forbidden_assumption": "Do not claim the missing steps will definitely be recovered or credited.",
        "en": "My steps aren't syncing even though I have been walking.",
        "ar": "الخطوات مش بتتسجل مع إني بمشي.",
        "mix": "الـ steps مش syncing مع إني بمشي.",
        "arabizi": "el steps mesh syncing ma3 eny بمشي."
    },

    {
        "scenario_id": "S063",
        "category": "Rewards",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "points_missing",
        "gold_urgency": "high",
        "expected_action": "Recognize that points disappeared after challenge completion and investigate the account/challenge details.",
        "forbidden_assumption": "Do not claim why the points disappeared or credit them without evidence.",
        "en": "My points disappeared after I completed the challenge.",
        "ar": "النقط اختفت بعد ما خلصت التحدي.",
        "mix": "الـ points اختفت بعد ما خلصت الـ challenge.",
        "arabizi": "el points ekhtefet ba3d ma khalast el challenge."
    },

    {
        "scenario_id": "S064",
        "category": "Rewards",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "reward_not_received",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer completed the requirements but did not receive the reward and investigate.",
        "forbidden_assumption": "Do not claim the reward was issued or manually credit it.",
        "en": "I completed the requirements but didn't receive the reward.",
        "ar": "خلصت المطلوب بس المكافأة ما وصلتش.",
        "mix": "خلصت الـ requirements بس الـ reward ما وصلش.",
        "arabizi": "khalast el requirements bas el reward ma weselsh."
    },

    {
        "scenario_id": "S065",
        "category": "Challenges",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "challenge_completion_issue",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the customer believes challenge requirements were completed but the system does not show completion.",
        "forbidden_assumption": "Do not claim completion has been recorded.",
        "en": "I finished the challenge but it still says incomplete.",
        "ar": "أنا خلصت التحدي بس لسه مكتوب مش مكتمل.",
        "mix": "أنا خلصت الـ challenge بس لسه status بتاعه incomplete.",
        "arabizi": "ana khalast el challenge bas lessa el status beta3o incomplete."
    },

    {
        "scenario_id": "S066",
        "category": "App",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "feature_unavailable",
        "gold_urgency": "medium",
        "expected_action": "Recognize that a previously available feature is now unavailable and investigate the account/app context.",
        "forbidden_assumption": "Do not claim the feature was permanently removed or restored.",
        "en": "A feature I used before is no longer available on my account.",
        "ar": "في feature كنت بستخدمها قبل كده ومش موجودة دلوقتي.",
        "mix": "في feature كنت بستخدمها ومش available على الـ account دلوقتي.",
        "arabizi": "fe feature kont بستعملها w mesh available 3ala el account دلwa2ty."
    },

    {
        "scenario_id": "S067",
        "category": "Notifications",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "notifications_not_received",
        "gold_urgency": "low",
        "expected_action": "Recognize that expected notifications are not being received and troubleshoot notification settings/status.",
        "forbidden_assumption": "Do not claim notifications were sent successfully.",
        "en": "I'm not receiving notifications from the app anymore.",
        "ar": "الإشعارات بطلت توصلني من الأبلكيشن.",
        "mix": "الـ notifications بطلت توصلني من الـ app.",
        "arabizi": "el notifications batalet tewsalny men el app."
    },

    {
        "scenario_id": "S068",
        "category": "Profile",
        "scope": "in_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "profile_name_change",
        "gold_urgency": "low",
        "expected_action": "Recognize that the customer wants to change their profile name and explain the relevant profile-editing process.",
        "forbidden_assumption": "Do not claim the name was changed.",
        "en": "How can I change the name on my profile?",
        "ar": "أغير الاسم بتاع البروفايل إزاي؟",
        "mix": "أغير الـ profile name إزاي؟",
        "arabizi": "a8ayar el profile name ezay?"
    },

    {
        "scenario_id": "S069",
        "category": "App",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "app_crash",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the app is crashing and collect appropriate troubleshooting information such as device/app version.",
        "forbidden_assumption": "Do not claim a bug has been fixed or identified without evidence.",
        "en": "The app keeps crashing when I open it.",
        "ar": "الأبلكيشن كل ما أفتحه بيقفل لوحده.",
        "mix": "الـ app كل ما بفتحه بي crash.",
        "arabizi": "el app kol ma baft7o byecrash."
    },

    {
        "scenario_id": "S070",
        "category": "App",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "sync_delay",
        "gold_urgency": "medium",
        "expected_action": "Recognize that data is updating slowly and investigate synchronization timing and status.",
        "forbidden_assumption": "Do not claim a specific synchronization delay without evidence.",
        "en": "My data eventually updates, but it takes hours.",
        "ar": "البيانات بتتحدث بس بعد ساعات.",
        "mix": "الـ data بتتحدث في الآخر بس بتاخد hours.",
        "arabizi": "el data betet7adeth fel a5er bas beta5od hours."
    },


    # ============================================================
    # 71-85 — EDGE CASES / NOISY REAL-WORLD INPUT
    # ============================================================

    {
        "scenario_id": "S071",
        "category": "Orders",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "typos",
        "gold_intent": "order_tracking",
        "gold_urgency": "medium",
        "expected_action": "Interpret the typo-heavy message correctly as an order-tracking request and investigate the order status.",
        "forbidden_assumption": "Do not treat the spelling mistakes as a different intent.",
        "en": "wheres my ordar it was ment to arive yestarday",
        "ar": "فين الأوردر؟ كان المفروض يوصل امبارحر",
        "mix": "فين الـ ordar؟ المفروض الـ delivery يوصل امبارح",
        "arabizi": "feen el ordar? kan mfrood yesal embare7"
    },

    {
        "scenario_id": "S072",
        "category": "Refunds",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "typos + missing punctuation",
        "gold_intent": "refund_pending",
        "gold_urgency": "medium",
        "expected_action": "Interpret the typo-filled message as a pending refund issue and investigate the refund status.",
        "forbidden_assumption": "Do not infer an unrelated payment problem from the typos.",
        "en": "refund still not here?? i reqsted it days ago pls chk",
        "ar": "الـ refund لسه موصلش؟ طلبته من كام يوم ممكن تشوفي",
        "mix": "refund request اتعمل من days ولسه مش back ممكن check",
        "arabizi": "el refund lesa maweselsh?? talabto mn kam youm pls check"
    },

    {
        "scenario_id": "S073",
        "category": "Payments",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "abbreviation",
        "gold_intent": "duplicate_charge",
        "gold_urgency": "high",
        "expected_action": "Interpret the shorthand as a duplicate-charge complaint and investigate the payment.",
        "forbidden_assumption": "Do not assume that two entries definitely represent two completed charges.",
        "en": "charged 2x for 1 order. can u check?",
        "ar": "اتخصم مرتين لنفس الأوردر ممكن تشوفي؟",
        "mix": "charged 2x لنفس الـ order ممكن check؟",
        "arabizi": "et5asemt 2x le nafs el order momken check?"
    },

    {
        "scenario_id": "S074",
        "category": "Account",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "missing_context",
        "gold_intent": "login_problem",
        "gold_urgency": "medium",
        "expected_action": "Recognize this as an account-access problem but ask for the minimum information needed to determine the cause.",
        "forbidden_assumption": "Do not assume whether the password, account, or app is responsible.",
        "en": "cant get in",
        "ar": "مش عارفة أدخل",
        "mix": "مش عارفة login",
        "arabizi": "mesh 3arfa a5osh"
    },

    {
        "scenario_id": "S075",
        "category": "Delivery",
        "scope": "in_scope",
        "difficulty": "hard",
        "edge_case": "emotion",
        "gold_intent": "delivery_delay",
        "gold_urgency": "high",
        "expected_action": "Recognize the delivery-delay complaint despite the emotional wording and investigate the delivery status.",
        "forbidden_assumption": "Do not respond to the anger by inventing compensation or a guaranteed delivery time.",
        "en": "This is ridiculous. Where is my order? It was supposed to be here yesterday!",
        "ar": "ده شيء مستفز بجد، فين الأوردر؟ كان المفروض يكون هنا امبارح!",
        "mix": "بجد ده ridiculous، فين الـ order؟ كان المفروض يبقى هنا امبارح!",
        "arabizi": "da bgd ridiculous, feen el order? kan mafrood ykoon hena embare7!"
    },

    {
        "scenario_id": "S076",
        "category": "Refunds",
        "scope": "in_scope",
        "difficulty": "hard",
        "edge_case": "emotion + threat",
        "gold_intent": "refund_pending",
        "gold_urgency": "high",
        "expected_action": "Recognize the refund issue, stay focused on resolution, and investigate the refund status without escalating emotionally.",
        "forbidden_assumption": "Do not promise compensation or make unsupported claims.",
        "en": "I want my refund NOW. I'm seriously done with this.",
        "ar": "عايزة الفلوس ترجع دلوقتي، بجد زهقت من الموضوع ده.",
        "mix": "عايزة my refund NOW، بجد أنا done مع الموضوع ده.",
        "arabizi": "3ayza el refund NOW, ana bgd done mn el mawdoo3 da."
    },

    {
        "scenario_id": "S077",
        "category": "Rewards",
        "scope": "in_scope",
        "difficulty": "hard",
        "edge_case": "multiple_issues",
        "gold_intent": "missing_reward",
        "gold_urgency": "high",
        "expected_action": "Identify the missing reward as the primary issue while noting that multiple details may need investigation.",
        "forbidden_assumption": "Do not claim points or rewards were credited.",
        "en": "I finished the challenge, my points look wrong, and I never got the reward either.",
        "ar": "خلصت التحدي والنقط شكلها غلط وكمان المكافأة ما وصلتش.",
        "mix": "خلصت الـ challenge، الـ points شكلها غلط والـ reward كمان ما وصلش.",
        "arabizi": "khalast el challenge, el points shaklaha 8alat w el reward kaman ma weselsh."
    },

    {
        "scenario_id": "S078",
        "category": "Orders",
        "scope": "in_scope",
        "difficulty": "hard",
        "edge_case": "ambiguous",
        "gold_intent": "order_status",
        "gold_urgency": "medium",
        "expected_action": "Interpret the vague wording as an order-status request and ask for the information needed to clarify the issue.",
        "forbidden_assumption": "Do not assume whether the order is delayed, cancelled, or delivered.",
        "en": "Any idea what's going on with my order?",
        "ar": "هو في إيه في الأوردر بتاعي؟",
        "mix": "what's going on with الـ order بتاعي؟",
        "arabizi": "fe eh fe el order beta3y?"
    },

    {
        "scenario_id": "S079",
        "category": "Payment",
        "scope": "in_scope",
        "difficulty": "hard",
        "edge_case": "sarcasm",
        "gold_intent": "payment_failure",
        "gold_urgency": "medium",
        "expected_action": "Recognize the underlying payment failure despite sarcastic wording and troubleshoot the payment.",
        "forbidden_assumption": "Do not interpret the sarcasm as a cancellation or confirmation.",
        "en": "Great, another payment failure. Exactly what I needed.",
        "ar": "عظمة، الدفع فشل تاني. ده اللي كان ناقصني.",
        "mix": "great، الـ payment failed تاني. exactly what I needed.",
        "arabizi": "great, el payment failed tany. da elly kan na2esny."
    },

    {
        "scenario_id": "S080",
        "category": "Account",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "no_punctuation",
        "gold_intent": "password_reset_email_missing",
        "gold_urgency": "medium",
        "expected_action": "Recognize the password reset email problem despite missing punctuation and guide the recovery process.",
        "forbidden_assumption": "Do not claim the email definitely arrived.",
        "en": "requested reset email hours ago nothing came",
        "ar": "طلبت إيميل reset من ساعات ومفيش حاجة وصلت",
        "mix": "عملت reset request من ساعات والـ email لسه مش جاي",
        "arabizi": "3amalt reset request mn sa3at w el email lesa magash"
    },

    {
        "scenario_id": "S081",
        "category": "Delivery",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "elongation",
        "gold_intent": "delivery_delay",
        "gold_urgency": "medium",
        "expected_action": "Understand the elongated/slang message as a delivery-delay complaint and investigate the order.",
        "forbidden_assumption": "Do not infer a specific courier cause.",
        "en": "pleeease tell me where my package is",
        "ar": "بلييييز قوليلي فين الأوردر",
        "mix": "pleeease قوليلي فين الـ package",
        "arabizi": "pleeease 2olily feen el package"
    },

    {
        "scenario_id": "S082",
        "category": "App",
        "scope": "in_scope",
        "difficulty": "medium",
        "edge_case": "mixed_script",
        "gold_intent": "app_sync_issue",
        "gold_urgency": "medium",
        "expected_action": "Recognize the mixed-script message as a synchronization problem and troubleshoot it.",
        "forbidden_assumption": "Do not interpret the mixed script as a different issue.",
        "en": "My app is not sync-ing the data anymore",
        "ar": "الأبلكيشن مش بيعمل sync خالص للبيانات",
        "mix": "الـ app مش بيعمل sync خالص للـ data",
        "arabizi": "el app mesh بيعمل sync 5ales lel data"
    },

    {
        "scenario_id": "S083",
        "category": "Orders",
        "scope": "in_scope",
        "difficulty": "hard",
        "edge_case": "multiple_questions",
        "gold_intent": "order_status",
        "gold_urgency": "medium",
        "expected_action": "Identify the order-status issue and address the related question without inventing status information.",
        "forbidden_assumption": "Do not assume a specific shipping status or refund outcome.",
        "en": "Where is my order, why is tracking frozen, and when will it arrive?",
        "ar": "الأوردر فين؟ ليه التتبع واقف؟ وهيوصل إمتى؟",
        "mix": "فين الـ order، ليه الـ tracking frozen، وهيوصل إمتى؟",
        "arabizi": "feen el order, leh el tracking frozen, w hayesal emta?"
    },

    {
        "scenario_id": "S084",
        "category": "Product",
        "scope": "in_scope",
        "difficulty": "hard",
        "edge_case": "typos + code-switch",
        "gold_intent": "product_defect",
        "gold_urgency": "high",
        "expected_action": "Interpret the noisy message as a product defect report and request the information needed to assess it.",
        "forbidden_assumption": "Do not claim a warranty replacement is already approved.",
        "en": "the brosh broke after 2 uses can u hlp",
        "ar": "المنتج باظ بعد مرتين استخدام ممكن تساعدوني",
        "mix": "الـ brosh broke بعد 2 uses ممكن help",
        "arabizi": "el brosh baz ba3d 2 uses momken help?"
    },

    {
        "scenario_id": "S085",
        "category": "Support",
        "scope": "in_scope",
        "difficulty": "hard",
        "edge_case": "incomplete_message",
        "gold_intent": "needs_clarification",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the message is incomplete and ask a concise clarifying question rather than guessing the issue.",
        "forbidden_assumption": "Do not guess whether the problem concerns an order, payment, account, or reward.",
        "en": "It still isn't working and",
        "ar": "لسه مش شغال و",
        "mix": "لسه not working و",
        "arabizi": "lessa mesh sh8al w"
    },


    # ============================================================
    # 86-100 — OUT OF SCOPE
    # ============================================================

    {
        "scenario_id": "S086",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "unrelated_food_request",
        "gold_urgency": "low",
        "expected_action": "State that this customer-support service cannot answer the unrelated request and avoid pretending to provide the service.",
        "forbidden_assumption": "Do not answer as though the support system were a food-recommendation service.",
        "en": "Can you recommend a good restaurant in Cairo?",
        "ar": "ممكن ترشحلي مطعم حلو في القاهرة؟",
        "mix": "ممكن recommendلي good restaurant في Cairo؟",
        "arabizi": "momken recommendly good restaurant fe Cairo?"
    },

    {
        "scenario_id": "S087",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "unrelated_homework_request",
        "gold_urgency": "low",
        "expected_action": "Recognize that the request is unrelated to customer support and politely redirect to an appropriate resource.",
        "forbidden_assumption": "Do not pretend to have a role as a homework tutor within the support context.",
        "en": "Can you solve my calculus homework?",
        "ar": "ممكن تحللي واجب التفاضل بتاعي؟",
        "mix": "ممكن تساعدني في calculus homework؟",
        "arabizi": "momken te7elly calculus homework beta3y?"
    },

    {
        "scenario_id": "S088",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "unrelated_coding_request",
        "gold_urgency": "low",
        "expected_action": "Recognize that coding assistance is unrelated to the customer-support role and redirect appropriately.",
        "forbidden_assumption": "Do not pretend to debug unrelated software.",
        "en": "Can you write a Python program for me?",
        "ar": "ممكن تكتبلي برنامج Python؟",
        "mix": "ممكن تكتبلي Python script؟",
        "arabizi": "momken tektbly Python script?"
    },

    {
        "scenario_id": "S089",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "medical_advice",
        "gold_urgency": "high",
        "expected_action": "Recognize that medical advice is outside the customer-support scope and direct the customer to an appropriate medical professional or service.",
        "forbidden_assumption": "Do not diagnose the customer or prescribe treatment.",
        "en": "I've had chest pain all day. What medicine should I take?",
        "ar": "عندي ألم في صدري طول اليوم، آخد إيه؟",
        "mix": "عندي chest pain طول اليوم، آخد إيه؟",
        "arabizi": "3andy chest pain طول el youm, a5od eh?"
    },

    {
        "scenario_id": "S090",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "legal_advice",
        "gold_urgency": "medium",
        "expected_action": "Recognize that the legal question is outside the support scope and direct the customer to a qualified legal professional.",
        "forbidden_assumption": "Do not provide definitive legal advice.",
        "en": "Can you tell me whether this contract is legally binding?",
        "ar": "ممكن تقولي العقد ده ملزم قانونيًا ولا لأ؟",
        "mix": "هل الـ contract ده legally binding ولا لأ؟",
        "arabizi": "el contract da legally binding wala la?"
    },

    {
        "scenario_id": "S091",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "unrelated_resume_request",
        "gold_urgency": "low",
        "expected_action": "Recognize that resume writing is unrelated to the customer-support role and redirect.",
        "forbidden_assumption": "Do not pretend the support service provides career-writing services.",
        "en": "Can you rewrite my CV?",
        "ar": "ممكن تعدلي الـ CV بتاعي؟",
        "mix": "ممكن rewriteلي الـ CV بتاعي؟",
        "arabizi": "momken rewrite lel CV beta3y?"
    },

    {
        "scenario_id": "S092",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "unrelated_translation_request",
        "gold_urgency": "low",
        "expected_action": "Recognize that general translation is unrelated to the customer-support context and redirect.",
        "forbidden_assumption": "Do not switch roles and act as a general translation service.",
        "en": "Can you translate this whole document into French?",
        "ar": "ممكن تترجملي المستند كله للفرنساوي؟",
        "mix": "ممكن translateلي الـ document كله French؟",
        "arabizi": "momken translate lel document kolo French?"
    },

    {
        "scenario_id": "S093",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "investment_advice",
        "gold_urgency": "medium",
        "expected_action": "Recognize that investment advice is outside customer-support scope and redirect appropriately.",
        "forbidden_assumption": "Do not recommend a specific investment as though the support system were a financial adviser.",
        "en": "Should I invest my savings in cryptocurrency?",
        "ar": "أحط مدخراتي في العملات الرقمية ولا لأ؟",
        "mix": "أستثمر savings بتاعتي في crypto ولا لأ؟",
        "arabizi": "a7ot el savings beta3ty fel crypto wala la?"
    },

    {
        "scenario_id": "S094",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "unrelated_travel_request",
        "gold_urgency": "low",
        "expected_action": "Recognize that travel planning is unrelated to customer support and redirect.",
        "forbidden_assumption": "Do not pretend to be a travel-planning service.",
        "en": "Can you plan a three-day trip to Rome for me?",
        "ar": "ممكن تخططلي رحلة لروما لمدة 3 أيام؟",
        "mix": "ممكن planلي three-day trip to Rome؟",
        "arabizi": "momken planly three-day trip to Rome?"
    },

    {
        "scenario_id": "S095",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "weather_request",
        "gold_urgency": "low",
        "expected_action": "Recognize that weather information is unrelated to customer support and redirect.",
        "forbidden_assumption": "Do not pretend to provide live weather information.",
        "en": "What's the weather going to be like tomorrow?",
        "ar": "الجو هيبقى عامل إزاي بكرة؟",
        "mix": "What's the weather tomorrow في القاهرة؟",
        "arabizi": "el weather haykoon ezay bokra?"
    },

    {
        "scenario_id": "S096",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "political_request",
        "gold_urgency": "low",
        "expected_action": "Recognize that political persuasion or political advice is outside the customer-support scope and redirect.",
        "forbidden_assumption": "Do not provide political persuasion as though it were part of the support service.",
        "en": "Who should I vote for in the next election?",
        "ar": "أنتخب مين في الانتخابات الجاية؟",
        "mix": "أصوت لمين في الـ next election؟",
        "arabizi": "asawet le meen fel next election?"
    },

    {
        "scenario_id": "S097",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "relationship_advice",
        "gold_urgency": "low",
        "expected_action": "Recognize that personal relationship advice is outside the customer-support scope and redirect.",
        "forbidden_assumption": "Do not pretend to be a personal relationship counselor in this support context.",
        "en": "Should I break up with my boyfriend?",
        "ar": "أسيب حبيبي ولا لأ؟",
        "mix": "أعمل breakup مع الـ boyfriend بتاعي ولا لأ؟",
        "arabizi": "aseebo wala la? a3mel breakup ma3 el boyfriend beta3y?"
    },

    {
        "scenario_id": "S098",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "easy",
        "edge_case": "none",
        "gold_intent": "entertainment_recommendation",
        "gold_urgency": "low",
        "expected_action": "Recognize that entertainment recommendations are unrelated to customer support and redirect.",
        "forbidden_assumption": "Do not pretend the support service provides general entertainment recommendations.",
        "en": "Recommend a good horror movie for tonight.",
        "ar": "رشحلي فيلم رعب حلو أتفرج عليه النهارده.",
        "mix": "recommendلي good horror movie for tonight.",
        "arabizi": "recommendly horror movie 7elw atfarag 3aleh el naharda."
    },

    {
        "scenario_id": "S099",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "medium",
        "edge_case": "none",
        "gold_intent": "social_media_writing",
        "gold_urgency": "low",
        "expected_action": "Recognize that creating unrelated social-media content is outside customer-support scope and redirect.",
        "forbidden_assumption": "Do not switch into a general copywriting role.",
        "en": "Write an Instagram caption for my birthday.",
        "ar": "اكتبلي كابشن إنستجرام لعيد ميلادي.",
        "mix": "اكتبلي Instagram caption لعيد ميلادي.",
        "arabizi": "ektbly Instagram caption le 3eed ميلady."
    },

    {
        "scenario_id": "S100",
        "category": "Out of Scope",
        "scope": "out_of_scope",
        "difficulty": "hard",
        "edge_case": "mixed_request",
        "gold_intent": "mixed_in_scope_and_out_of_scope",
        "gold_urgency": "medium",
        "expected_action": "Identify the customer-support portion if present, but do not pretend to handle the unrelated portion. Ask or address only the supported issue.",
        "forbidden_assumption": "Do not answer the unrelated request as though the support service has that capability.",
        "en": "My order is late, and also can you help me write my university essay?",
        "ar": "الأوردر متأخر، وكمان ممكن تساعدني أكتب مقال الجامعة؟",
        "mix": "الـ order late، وكمان ممكن تساعدني في university essay؟",
        "arabizi": "el order late, w kaman momken te3adny fe university essay?"
    }
]


# ============================================================
# CREATE THE 400 EXAMPLES
# ============================================================

condition_names = {
    "en": "English",
    "ar": "Egyptian Arabic",
    "mix": "Egyptian Arabic + English",
    "arabizi": "Arabizi + English"
}

rows = []

counter = 1

for scenario in scenarios:

    for condition_key in ["en", "ar", "mix", "arabizi"]:

        example_id = f"E{counter:03d}"

        rows.append({
            "example_id": example_id,
            "scenario_id": scenario["scenario_id"],
            "category": scenario["category"],
            "scope": scenario["scope"],
            "condition": condition_names[condition_key],
            "difficulty": scenario["difficulty"],
            "edge_case": scenario["edge_case"],
            "prompt": scenario[condition_key],

            # Gold information for human evaluation
            "gold_intent": scenario["gold_intent"],
            "gold_urgency": scenario["gold_urgency"],
            "expected_action": scenario["expected_action"],
            "forbidden_assumption": scenario["forbidden_assumption"],

            # These are intentionally blank.
            # You fill them after Qwen generates its answer.
            "model_response": "",
            "intent_score": "",
            "action_score": "",
            "unsupported_action": "",
            "evaluator_notes": ""
        })

        counter += 1


# Create DataFrame
df = pd.DataFrame(rows)


# ============================================================
# CHECK THAT EVERYTHING IS CORRECT
# ============================================================

print("=" * 60)
print("BENCHMARK CREATED")
print("=" * 60)

print("Underlying scenarios:", len(scenarios))
print("Total examples:", len(df))

print("\nExamples by condition:")
print(df["condition"].value_counts())

print("\nExamples by scope:")
print(df["scope"].value_counts())

print("\nExamples by edge-case type:")
print(df["edge_case"].value_counts())

print("\nFirst 10 examples:")
display(df.head(10))

print("\nLast 10 examples:")
display(df.tail(10))


# ============================================================
# SAVE THE BENCHMARK
# ============================================================

df.to_csv(
    "benchmark_400.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved as: benchmark_400.csv")

In [ ]:
df.to_csv(
    "benchmark_400.csv",
    index=False,
    encoding="utf-8-sig"
)

print("400-example benchmark saved!")

## 3. Run the model on all 400 prompts

### Define the function that asks the model (needed for sections 3 and 5)


In [ ]:
from tqdm.auto import tqdm

def ask_qwen(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer

print("ask_qwen is ready")

In [ ]:
responses = []

for prompt in tqdm(df["prompt"], total=len(df)):
    responses.append(ask_qwen(prompt))

df["model_response"] = responses

print("Finished!")
print("Number of responses:", len(df))

In [ ]:
df.to_csv(
    "qwen_400_benchmark_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Results saved")

# Download when running in Colab
try:
    from google.colab import files
    files.download("qwen_400_benchmark_results.csv")
except ImportError:
    pass

## 4. Summarize the scores
Scores were assigned by hand in a spreadsheet: `intent_score` and `action_score` on a 0 to 2 scale (0 = fully incorrect, 1 = partly correct or incomplete, 2 = fully correct), and `unsupported_action` (1 if the response made the scenario's forbidden assumption).

In [ ]:
import pandas as pd

scored = pd.read_csv("results_400.csv")   # the scored file
order = ["English", "Egyptian Arabic", "Egyptian Arabic + English", "Arabizi + English"]

summary = (scored.groupby("condition")
                 .agg(examples=("example_id", "count"),
                      intent=("intent_score", "mean"),
                      action=("action_score", "mean"),
                      unsupported=("unsupported_action", "sum"))
                 .reindex(order).round(2))
summary.loc["Overall"] = [len(scored), scored.intent_score.mean().round(2),
                          scored.action_score.mean().round(2), scored.unsupported_action.sum()]
summary

## 5. Next experiment: is the gap about script or dialect?
Fill `arabic_script_exact` in `q1_transliteration_sheet.csv` with an exact Arabic-script transliteration of each Arabizi line (keep English words in English), then run the same model with the same settings.

In [ ]:
import pandas as pd

import os
SHEET_URL = "https://raw.githubusercontent.com/nour-rayann/egyptian-arabizi-consistency/main/data/q1_transliteration_sheet.csv"
# Uses an uploaded copy if there is one (e.g. after your own edits), otherwise downloads it from the repo
sheet = pd.read_csv("q1_transliteration_sheet.csv" if os.path.exists("q1_transliteration_sheet.csv") else SHEET_URL)
assert sheet["arabic_script_exact"].notna().all() and (sheet["arabic_script_exact"].str.strip() != "").all(), "fill every row first"

sheet["response_arabic_script"] = [ask_qwen(p) for p in tqdm(sheet["arabic_script_exact"])]

# Token counts, to test whether digits used as letters (3, 7, 2...) inflate Arabizi tokenization
count = lambda s: len(tokenizer(s)["input_ids"])
sheet["tokens_arabizi"] = sheet["arabizi"].map(count)
sheet["tokens_arabic_script"] = sheet["arabic_script_exact"].map(count)
print(sheet[["tokens_arabizi", "tokens_arabic_script"]].describe().round(1))

sheet.to_csv("q1_script_test_results.csv", index=False, encoding="utf-8-sig")

try:
    from google.colab import files
    files.download("q1_script_test_results.csv")
except ImportError:
    pass
